# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hafsa-SE/flyrank-assignment/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

**The FlyRank case study.** FlyRank publishes content straight into client websites and then
watches search performance to catch problems -- the core promise is content that gets managed,
not abandoned after publishing. The trouble is the same in almost every portfolio: a page ranks
well, then quietly decays -- rankings slip, clicks drop -- and most teams notice too late. Out
of thousands of pages, the one decision that actually matters is **which page a human should
fix first.**

FlyRank's product already answers this today with hand-written rules: a health score,
quick-win tags, needs-attention flags -- if-this-then-that logic with hand-picked thresholds.
These rules work and run in production, but they run out exactly where signals get numerous,
tangled, and shifting -- and nothing has replaced them with something learned from data. That
gap is this paper's question, restated precisely:

**Which pages in a content portfolio should a human review first for a possible SEO decline,
and why -- and can a model trained on real signals beat the hand-written rule it's meant to
replace?** This supports a weekly triage decision for a content team lead: given limited
review time, rank pages by (a) how likely they are to be declining and (b) how much
click-equivalent value is already riding on them -- so the highest-value, highest-risk pages
get looked at first, not last.

Per FlyRank's own house rule, the existing product flags are treated strictly as an **output to
beat**, never as a model input -- using them as a feature would just launder an existing
decision back into "new" results (see the leakage audit in Methodology and Week 6's notebook).


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 130)
print("Research question: rank pages for SEO-decline review, using only pre-decision signals")
print("(no future data, no label-derived features) -- see Methodology for the exact label.")


Research question: rank pages for SEO-decline review, using only pre-decision signals
(no future data, no label-derived features) -- see Methodology for the exact label.


## 2. Data

**Source:** the FlyRank ML Internship dataset (anonymized content-performance release),
`data/raw/content_refresh_anonymized.csv` -- one row per content page, aggregated search and
engagement metrics over a 90-day window, plus a 30-day-vs-previous-30-day trend comparison.
Full field definitions are in `docs/data-dictionary.md`.

**Size:** 30,000 rows across 32 anonymized clients (portfolios), spanning a range of content
ages, intents, and position tiers.

**What was excluded and why:** rows are anonymized at the client level (no client names, no
raw URLs, no query text anywhere in this analysis or the paper). Any column that defines or
leaks the trend label (`trend_direction`, `trend_pct`, and the `*_last_30d` / `*_prev_30d`
comparison-window columns) was excluded from every model's feature set -- kept only as the
evaluation target, never as an input (see Methodology and the leakage audit below).


In [2]:
DATA_PATH = Path("../../data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(DATA_PATH)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(f"Rows: {len(df):,}  |  Clients: {df['client_id'].nunique()}")
print(f"Label positive rate (base rate, 'declining'): {df['is_declining_label'].mean():.3f}")
print(f"Position tiers: {sorted(df['position_tier'].dropna().unique().tolist())}")
print(f"Content types: {sorted(df['content_type'].dropna().unique().tolist())}")


Rows: 30,000  |  Clients: 32
Label positive rate (base rate, 'declining'): 0.542
Position tiers: ['deep', 'page_1', 'page_3_5', 'striking', 'top_3']
Content types: ['comparison article', 'feedly article', 'keyword article']


## 3. Methodology

**Label:** `is_declining_label` = 1 if `trend_direction == "down"` (a 30-day-vs-previous-30-day
comparison), else 0. This is the same trend signal the source research paper's own
freshness-window findings are built on.

**Features:** 18 numeric signals (search volume, competition, word/char counts, days with
impressions/sessions, content age, days since last update, CTR, average position, engagement
rate, scroll rate, AI-traffic share, and log-scaled 90-day totals) plus 8 categorical tiers
(competition level, content type, intent, age/freshness/word-count/impression/position tiers),
one-hot encoded. **Explicitly excluded:** `trend_direction`, `trend_pct`, and every
`*_last_30d` / `*_prev_30d` column -- these define or leak the label.

**Baseline (Week 4):** a transparent, non-ML rule -- score visible pages (real position,
>=500 impressions/90d) by how far their CTR sits below their own position-tier's median CTR,
weighted by demand (`log1p(impressions_90d)`). One reason code, no fitted weights.

**Model:** Random Forest (200 trees, max depth 10, `class_weight="balanced_subsample"`),
compared against Logistic Regression and a shallow Decision Tree; Random Forest won on
precision@50 and is reported as the primary model.

**Validation design:** client-grouped train/test split (20% of the 32 clients held out
entirely) -- not a random row split. Rows from the same client share hidden structure (site
maturity, content style), so a random split would let a model partly recognize clients it has
already seen, inflating its apparent skill (demonstrated quantitatively in Results below).

**Leakage checks:** (1) forbidden-column check -- confirm the trend/label-derived columns
never enter the feature matrix; (2) correlation scan -- confirm no feature is suspiciously
close (|r| > 0.9) to the label itself.


In [3]:
NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]
CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier",
    "impression_tier", "position_tier",
]
for col in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    df[f"log_{col}"] = np.log1p(df[col].clip(lower=0))
NUMERIC_FEATURES += ["log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d"]

FORBIDDEN = {"trend_direction", "trend_pct", "is_declining_label",
             "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
             "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"}
used = set(NUMERIC_FEATURES + CATEGORICAL_FEATURES)
assert not (used & FORBIDDEN), f"Leakage: {used & FORBIDDEN}"

numeric_frame = df[NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
categorical_frame = df[CATEGORICAL_FEATURES].fillna("unknown").astype(str)
encoded = pd.get_dummies(categorical_frame, prefix=CATEGORICAL_FEATURES, dtype=float)
X = pd.concat([numeric_frame.reset_index(drop=True), encoded.reset_index(drop=True)], axis=1)
y = df["is_declining_label"].astype(int)

# --- Leakage check: correlation scan ---
numeric_corr = numeric_frame.assign(is_declining_label=y).corr(numeric_only=True)["is_declining_label"].drop("is_declining_label")
max_corr = numeric_corr.abs().max()
print(f"Feature matrix: {X.shape}")
print(f"Max |correlation| of any numeric feature with the label: {max_corr:.3f} (no feature exceeds 0.9)")
assert max_corr < 0.9, "Leakage: a feature is near-perfectly correlated with the label."
print("Leakage checks passed: no forbidden columns used, no suspiciously high correlation.")


Feature matrix: (30000, 52)
Max |correlation| of any numeric feature with the label: 0.190 (no feature exceeds 0.9)
Leakage checks passed: no forbidden columns used, no suspiciously high correlation.


## 4. Results (vs baseline)

Same held-out test rows, same metrics, for the rule-based baseline and the trained models.
Then the validation-design check: the SAME Random Forest re-scored under a plain random row
split, to show numerically why the grouped split is the honest one.


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 42

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    top = np.asarray(y_true)[order[:k]]
    return float(top.mean()) if len(top) else float("nan")

# --- Client-grouped split (the honest one) ---
client_series = df["client_id"].astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
shuffled_clients = rng.permutation(unique_clients)
n_test_clients = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:n_test_clients])
grouped_test_mask = client_series.isin(test_clients).to_numpy()
train_idx, test_idx = np.where(~grouped_test_mask)[0], np.where(grouped_test_mask)[0]

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# --- Baseline, recomputed on this exact split ---
measurable = df.iloc[train_idx]
measurable = measurable[(measurable["avg_position"] > 0) & (measurable["impressions_90d"] >= 500)]
tier_benchmark_ctr = measurable.groupby("position_tier", observed=True)["ctr"].median()
ctr_gap = (df["position_tier"].map(tier_benchmark_ctr) - df["ctr"]).clip(lower=0)
is_visible = (df["avg_position"] > 0) & (df["impressions_90d"] >= 500)
baseline_score_full = np.where(is_visible, ctr_gap * np.log1p(df["impressions_90d"]), 0.0)
baseline_test_scores = baseline_score_full[test_idx]

models = {
    "logistic_regression": Pipeline([("scaler", StandardScaler()),
                                      ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))]),
    "decision_tree": DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, class_weight="balanced", random_state=RANDOM_STATE),
    "random_forest": RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25,
                                             class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE),
}
results = {}
fitted = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted[name] = model
    proba = model.predict_proba(X_test)[:, 1]
    results[name] = {"precision_at_20": precision_at_k(y_test, proba, 20),
                      "precision_at_50": precision_at_k(y_test, proba, 50),
                      "roc_auc": roc_auc_score(y_test, proba)}
results["baseline_rule"] = {"precision_at_20": precision_at_k(y_test, baseline_test_scores, 20),
                             "precision_at_50": precision_at_k(y_test, baseline_test_scores, 50),
                             "roc_auc": roc_auc_score(y_test, baseline_test_scores)}
comparison = pd.DataFrame(results).T
comparison["base_rate"] = y_test.mean()
comparison = comparison.sort_values("precision_at_50", ascending=False)
print(f"Test: {len(y_test):,} rows, {n_test_clients} held-out clients, base rate {y_test.mean():.3f}\n")
print(comparison.round(3).to_string())

best_name = comparison.index[0]
best_model = fitted[best_name]

# --- Validation-design check: same model, random row split instead of grouped ---
rng2 = np.random.default_rng(RANDOM_STATE)
shuffled_rows = rng2.permutation(len(df))
n_test_rows = int(round(len(df) * 0.2))
rand_test_idx, rand_train_idx = shuffled_rows[:n_test_rows], shuffled_rows[n_test_rows:]
overlap_clients = len(set(df["client_id"].iloc[rand_train_idx]) & set(df["client_id"].iloc[rand_test_idx]))

rf_random_split = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25,
                                          class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE)
rf_random_split.fit(X.iloc[rand_train_idx], y.iloc[rand_train_idx])
proba_random = rf_random_split.predict_proba(X.iloc[rand_test_idx])[:, 1]
p50_random = precision_at_k(y.iloc[rand_test_idx], proba_random, 50)
p50_grouped = comparison.loc["random_forest", "precision_at_50"]

print(f"\nValidation design check -- same model, two splits:")
print(f"  Random row split  ({overlap_clients}/32 clients leak into both sides): precision@50 = {p50_random:.3f}")
print(f"  Client-grouped split (0 clients overlap):                              precision@50 = {p50_grouped:.3f}")
print(f"  Gap: {p50_random - p50_grouped:+.3f} -- the random split's number is inflated by client leakage, not real skill.")


Test: 2,325 rows, 6 held-out clients, base rate 0.391

                     precision_at_20  precision_at_50  roc_auc  base_rate
random_forest                   0.90             0.78    0.747      0.391
baseline_rule                   0.85             0.66    0.555      0.391
decision_tree                   0.65             0.54    0.698      0.391
logistic_regression             0.35             0.40    0.700      0.391



Validation design check -- same model, two splits:
  Random row split  (31/32 clients leak into both sides): precision@50 = 0.980
  Client-grouped split (0 clients overlap):                              precision@50 = 0.780
  Gap: +0.200 -- the random split's number is inflated by client leakage, not real skill.


## 5. Limitations

- **One portfolio, one 90-day snapshot.** Findings are observed/measured on this dataset;
  not a universal SEO law.
- **Cross-sectional, not causal.** "High risk_score" is an association the model learned, not
  proof that a refresh will fix the page.
- **Small held-out set.** precision@50 = 0.78 comes from ~2,300 rows across 6 held-out
  clients -- a real, honestly-obtained number, but a small one; treat it as directional.
- **The model doesn't diagnose root cause.** A `refresh_priority` flag can mean "content is
  stale" or, in some cases, "technical/indexing problem" -- error analysis in Week 5 found
  exactly this pattern among the model's most confident misses (near-zero-CTR, decent-position
  pages). A human must confirm the cause before acting.
- **Value is a proxy.** `clicks_90d x cpc` estimates click-equivalent value; it is not
  confirmed revenue.


In [5]:
# Concrete evidence behind the "doesn't diagnose root cause" limitation.
proba_best = best_model.predict_proba(X_test)[:, 1]
test_frame = df.iloc[test_idx][["content_id", "position_tier", "impressions_90d", "ctr"]].copy()
test_frame["y_true"] = y_test.to_numpy()
test_frame["y_prob"] = proba_best
confident_wrong = test_frame[((test_frame["y_prob"] > 0.7) & (test_frame["y_true"] == 0))]
low_ctr_share = (confident_wrong["ctr"] < 0.1).mean() if len(confident_wrong) else float("nan")
print(f"Of {len(confident_wrong)} high-confidence 'declining' predictions that were actually stable,")
print(f"{low_ctr_share:.0%} had near-zero CTR (<0.1%) despite a real position -- consistent with a")
print("technical/indexing issue being mistaken for a content-decline signal. Root-cause check needed.")


Of 37 high-confidence 'declining' predictions that were actually stable,
73% had near-zero CTR (<0.1%) despite a real position -- consistent with a
technical/indexing issue being mistaken for a content-decline signal. Root-cause check needed.


## 6. Ranked recommendations

The Week-7 action playbook, regenerated here: five archetypes (freshness/CTR-based, independent
of the model), each with a default action, ranked within-archetype by `risk_score x
log(value_score)`. Full detail and the no-go list live in `work/notebooks/w07_action_playbook.ipynb`.


In [6]:
final_model = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25,
                                      class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE)
final_model.fit(X, y)
risk_score = final_model.predict_proba(X)[:, 1]
value_score = (df["clicks_90d"].fillna(0) * df["cpc"].fillna(0)).clip(lower=0)

is_visible_full = (df["avg_position"] > 0) & (df["impressions_90d"] >= 500)
is_stale = df["days_since_last_update"] >= 90
measurable_full = df[is_visible_full]
tier_benchmark_full = measurable_full.groupby("position_tier", observed=True)["ctr"].median()
ctr_below = df["ctr"] < df["position_tier"].map(tier_benchmark_full)

archetype = np.select(
    [~is_visible_full, is_visible_full & is_stale, is_visible_full & ~is_stale & ctr_below,
     is_visible_full & ~is_stale & ~ctr_below & (df["position_tier"].isin(["deep", "page_3_5"])),
     is_visible_full & ~is_stale & ~ctr_below],
    ["monitor_only", "refresh_priority", "ctr_fix_candidate", "low_priority_deep", "protect_and_expand"],
    default="monitor_only",
)
df["archetype"] = archetype
ACTION_BY_ARCHETYPE = {"refresh_priority": "refresh_content", "ctr_fix_candidate": "fix_snippet_and_meta",
                        "protect_and_expand": "leave_or_expand_thin_sections",
                        "low_priority_deep": "low_priority_review", "monitor_only": "monitor_no_action"}
df["action"] = df["archetype"].map(ACTION_BY_ARCHETYPE)
df["risk_score"] = risk_score
priority = np.where(df["archetype"] == "monitor_only", 0.0, risk_score * np.log1p(value_score))
df["playbook_priority"] = priority
df["playbook_rank"] = df["playbook_priority"].rank(method="first", ascending=False).astype(int)

print("Archetype distribution (full portfolio):")
print(df["archetype"].value_counts())
print("\nTop 5 of the ranked recommendation queue:")
print(df.sort_values("playbook_rank")[["playbook_rank", "archetype", "action", "position_tier", "risk_score"]].head(5).to_string(index=False))


Archetype distribution (full portfolio):
archetype
monitor_only          13274
refresh_priority       6575
ctr_fix_candidate      4723
protect_and_expand     4006
low_priority_deep      1422
Name: count, dtype: int64

Top 5 of the ranked recommendation queue:
 playbook_rank          archetype                        action position_tier  risk_score
             1  ctr_fix_candidate          fix_snippet_and_meta        page_1    0.648313
             2   refresh_priority               refresh_content        page_1    0.765498
             3   refresh_priority               refresh_content        page_1    0.705110
             4 protect_and_expand leave_or_expand_thin_sections      striking    0.720706
             5 protect_and_expand leave_or_expand_thin_sections        page_1    0.615979


## 7. Artifacts the paper embeds

Three charts, exported to `work/figures/` (committed) for the deployed paper; a metrics JSON
to `work/outputs/` (committed, the receipts); the full ranked queue CSV to `work/outputs/`
(gitignored by design -- regenerates on every run).


In [7]:
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig_dir = Path("../figures")
fig_dir.mkdir(parents=True, exist_ok=True)
out_dir = Path("../outputs")
out_dir.mkdir(parents=True, exist_ok=True)

# --- Chart 1: model vs baseline ---
chart1 = comparison[["precision_at_20", "precision_at_50"]].sort_values("precision_at_50")
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(chart1))
width = 0.35
ax.barh(x - width/2, chart1["precision_at_20"], height=width, label="precision@20", color="#4a7c9e")
ax.barh(x + width/2, chart1["precision_at_50"], height=width, label="precision@50", color="#2f6f4f")
ax.set_yticks(x); ax.set_yticklabels(chart1.index)
ax.axvline(y_test.mean(), color="gray", linestyle="--", linewidth=1, label=f"base rate ({y_test.mean():.2f})")
ax.set_xlabel("Precision"); ax.set_title("Model vs. baseline, client-grouped held-out test")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.savefig(fig_dir / "fig1_model_vs_baseline.png", dpi=120); plt.close(fig)

# --- Chart 2: honest split vs random split ---
fig, ax = plt.subplots(figsize=(6, 4.5))
labels = ["Random row split\n(client leakage)", "Client-grouped split\n(honest)"]
values = [p50_random, p50_grouped]
colors = ["#c0392b", "#2f6f4f"]
bars = ax.bar(labels, values, color=colors)
for b, v in zip(bars, values):
    ax.text(b.get_x() + b.get_width()/2, v + 0.01, f"{v:.2f}", ha="center", fontweight="bold")
ax.set_ylabel("Random Forest precision@50"); ax.set_ylim(0, 1.05)
ax.set_title("Validation design matters: same model, two splits")
plt.tight_layout(); plt.savefig(fig_dir / "fig2_split_design.png", dpi=120); plt.close(fig)

# --- Chart 3: archetype distribution (recreated fresh here for reproducibility) ---
counts = df["archetype"].value_counts().sort_values()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(counts.index, counts.values, color="#2f6f4f")
for i, v in enumerate(counts.values):
    ax.text(v, i, f"  {v:,}", va="center")
ax.set_xlabel("Number of pages"); ax.set_title("Content action playbook: pages by archetype")
plt.tight_layout(); plt.savefig(fig_dir / "fig3_archetype_distribution.png", dpi=120); plt.close(fig)

print("Saved: fig1_model_vs_baseline.png, fig2_split_design.png, fig3_archetype_distribution.png")

# --- Metrics JSON (the receipts the paper's numbers trace back to) ---
capstone_metrics = {
    "n_rows": int(len(df)), "n_clients": int(df["client_id"].nunique()),
    "held_out_clients": int(n_test_clients), "held_out_rows": int(len(y_test)),
    "held_out_base_rate": round(float(y_test.mean()), 3),
    "comparison_table": comparison.round(4).to_dict(orient="index"),
    "split_design_check": {"random_split_precision_at_50": round(float(p50_random), 4),
                            "grouped_split_precision_at_50": round(float(p50_grouped), 4),
                            "clients_leaking_in_random_split": int(overlap_clients)},
    "archetype_counts": df["archetype"].value_counts().to_dict(),
}
with open(out_dir / "capstone_metrics.json", "w") as f:
    json.dump(capstone_metrics, f, indent=2)
print("Saved capstone_metrics.json")

# --- Full ranked queue CSV (gitignored, regenerates every run) ---
df.sort_values("playbook_rank")[["playbook_rank", "content_id", "client_id", "archetype", "action",
                                   "risk_score", "playbook_priority", "position_tier"]].to_csv(
    out_dir / "capstone_ranked_queue.csv", index=False)
print("Saved capstone_ranked_queue.csv")


Saved: fig1_model_vs_baseline.png, fig2_split_design.png, fig3_archetype_distribution.png
Saved capstone_metrics.json
Saved capstone_ranked_queue.csv


## 8. Repurposing this work (ML-12)

**5-minute demo outline:**
1. (30s) The question -- FlyRank's product already flags declining pages with hand-written
   rules; which pages should a human review first, and can a trained model beat that rule?
2. (60s) The baseline -- one transparent rule, standing in for FlyRank's own flag logic,
   honestly evaluated.
3. (90s) The model -- Random Forest vs. baseline on a client-grouped split; show the
   before/after split-design chart, since that's the most persuasive result in the whole project.
4. (60s) Limitations -- say them before anyone asks.
5. (60s) The playbook -- five archetypes, one no-go list, live numbers.

**Social-post cut:**
> FlyRank's product flags declining pages with hand-written rules today. I built a model to
> see if a trained one could do better -- on 30K real content pages across 32 clients. The
> honest number (client-grouped validation): 78% precision in the top 50 flagged pages vs. 66%
> for the rule-based baseline and a 39% base rate on held-out clients. The more interesting
> number: the same model scored under a naive random split looked 20 points better -- a
> reminder that *how* you validate matters as much as *what* you build. Full writeup +
> reproducible notebooks linked below.

**3-sentence employer-facing summary:**
I built and validated a content-decline risk model for FlyRank's content-monitoring problem
(which of thousands of pages should a human review first) on a 30,000-row, 32-client SEO
performance dataset, comparing a transparent rule-based baseline -- standing in for FlyRank's
existing hand-written flags -- against Logistic Regression, Decision Tree, and Random Forest
under a client-grouped validation split designed to prevent leakage. The final model reached
0.78 precision@50 (vs. 0.66 baseline, 0.39 base rate) on held-out clients, and I quantified how
much a naive random split would have overstated that number (0.98). The project ships as a
public research paper with a ranked, human-reviewed action playbook, full leakage/validation
audits, and every notebook reproducible from the source repo.


In [8]:
print("Demo outline, social-post cut, and employer summary written above (markdown cell).")
print("These are repurposed directly from the Results/Limitations numbers computed in this notebook --")
print("no new numbers invented here.")


Demo outline, social-post cut, and employer summary written above (markdown cell).
These are repurposed directly from the Results/Limitations numbers computed in this notebook --
no new numbers invented here.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
